# Fitted ECD-Threshold Comparison (Analysis A2)

## Protocol

**Status:** fixed on 2026-10-16, before any model in this notebook was run. Follow-up to analysis A (`10_beyond_ecd_comparison.ipynb`), recorded in git before execution.

### 1. Framing and disclosure

This is a **follow-up analysis**. It is not independent confirmation, and it is motivated by results already seen:

- Notebook 10 (pre-specified) found that the tabular and hybrid models significantly outperformed an **ECD-only Random Forest** (macro F1 +0.048 and +0.081, Holm p = 0.030 and 0.006).
- In the same notebook, a fixed two-threshold ECD rule scored 0.797 against 0.707 for the ECD-only Random Forest (descriptive). This suggested that the Random Forest was a weak comparator for "ECD alone". The rule's thresholds (2250/2500) were inferred from 2023 records after inspection.
- The baseline in this notebook (a depth-2 tree, i.e. at most three cut points) was chosen **knowing** that a two-threshold structure performs well on these tissues.

This analysis uses the same tissues, folds and models as notebooks 09–10. Its multiplicity correction covers only the two comparisons below, not notebooks 09–11 jointly.

### 2. Research question

Do the tabular and hybrid models improve on an ECD-only threshold model whose cut points are fitted within each training fold, when all are evaluated on the same unseen donors?

### 3. Cohort

Identical to notebook 10; the notebook asserts these counts and stops if any differ.

| Set | Expected |
|---|---|
| Primary cohort (non-2023, image-matched, year from `Tissue_ID`) | 551 tissues, 312 donors |
| **Evaluation set** (primary cohort with ECD) | **470 tissues, 280 donors**; A+ 244, A 159, B 67; years 2020: 78, 2021: 11, 2022: 54, 2024: 155, 2025: 172 |

### 4. Models

**ECD-only tree (new baseline):**

`DecisionTreeClassifier(max_depth=2, min_samples_leaf=10, class_weight="balanced", criterion="gini", random_state=42)`, with `ECD_CD_cells_per_mm2` as the only input.

- Trained on training-fold tissues with ECD.
- `max_depth=2` is the smallest depth that can express two thresholds separating three grades. The descriptive rule has exactly this form.
- `min_samples_leaf=10` prevents cut points defined by a handful of tissues.
- `class_weight="balanced"` matches the other models and the macro-averaged metric.
- These values are fixed here and are **not tuned**.

**Comparators (unchanged from notebook 10):**

| Model | Inputs | Training tissues |
|---|---|---|
| Tabular | 24 features, Random Forest as notebooks 07–10 | All training-fold tissues of the primary cohort |
| Hybrid | 24 features + frozen ResNet-50 → PCA 16, same Random Forest | All training-fold tissues of the primary cohort |
| ECD-only Random Forest (descriptive) | ECD alone, same Random Forest | Training-fold tissues with ECD |

**Reproducibility check:** the Tabular, Hybrid and ECD-only Random Forest fold-level scores must match `beyond_ecd_fold_scores.csv` from notebook 10, to within floating-point rounding (absolute difference < 1e-9). If they do not, the notebook stops and the discrepancy is recorded in the deviations log.

### 5. Validation

- Existing folds `Fold_r0` … `Fold_r4` from `donor_grouped_folds.csv`, restricted to the primary cohort. All fitting uses the training fold only.
- Metrics are computed on the evaluation-set tissues in each held-out fold.
- Checks before fitting:
  - no donor in both training and test;
  - every primary-cohort tissue is tested once per repeat;
  - every held-out fold has evaluation-set tissues of each grade.

### 6. Comparisons and metrics

| Role | Comparison | Metric |
|---|---|---|
| **Primary P1** | Tabular − ECD-only tree | Macro F1, per fold |
| **Primary P2** | Hybrid − ECD-only tree | Macro F1, per fold |
| Secondary | P1 and P2 | Balanced accuracy, accuracy, grade B recall (unadjusted) |
| Descriptive | ECD-only tree vs ECD-only Random Forest and vs the 2250/2500 rule | Macro F1 |
| Descriptive | All models | Per-grade precision and recall, pooled over repeats |
| Descriptive | ECD-only tree | Cut points learned in each of the 25 training folds, with their range |

### 7. Statistical analysis

- For P1 and P2: mean paired fold-level difference, corrected 95% CI and corrected repeated-CV t-test (Nadeau & Bengio, 2003; Bouckaert & Frank, 2004), 24 degrees of freedom.
  - The variance correction uses n_test / n_train from mean primary-cohort fold sizes.
- **Multiplicity:** Holm correction across P1 and P2. Confidence intervals are not multiplicity-adjusted.
- **Per-year (descriptive):** pooled difference by year and with each year excluded in turn; 2021 (11 tissues) flagged as too small.

### 8. Interpretation rules

Fixed before running and applied in code.

**Each primary comparison:**

| Result | Reported as |
|---|---|
| Difference > 0 and Holm-adjusted p < 0.05 | Model improved on the fitted ECD-only tree |
| Difference < 0 and Holm-adjusted p < 0.05 | Model performed worse than the fitted ECD-only tree |
| Otherwise | Inconclusive: no evidence of a difference from the fitted ECD-only tree |

**Answer to the research question:**

- *Evidence of improvement beyond a fitted ECD-only threshold model (follow-up analysis)*: at least one comparison is improved.
- *Improvement beyond ECD alone not established*: neither is improved and neither is worse.
- *Models performed worse than a fitted ECD-only threshold model*: either is worse. The specific model or models are named.

**Qualifiers:**

- **Grade B recall:** for each comparison, if the mean fold-level grade B recall difference is below −0.02, the result is reported with the reduced B recall and that model is not recommended over the ECD-only tree. As before, −0.02 is a project-defined decision rule, not a clinically validated threshold.
- **Per-year:** if a positive pooled difference becomes zero or negative when any single year is excluded, the result is reported as dependent on that year.

**Wording for reporting:**

- In all cases, notebook 10's result (improvement over an ECD-only Random Forest) is reported alongside this one, not replaced by it.
- This analysis is described as a follow-up motivated by notebook 10, not as independent confirmation.

**Other rules:**

- **No changes after results:** no other depths, leaf sizes, learners, cut-point methods, cohorts or metrics. Any further analysis is labelled post hoc.
- **Errors:** a code error may be fixed and the notebook rerun only if the fix does not change a protocol choice. Every rerun is logged in the deviations log with its reason.

# Analysis

Implements the protocol above without changes. Section numbers refer to the protocol.

In [1]:
import hashlib
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path("../../scripts").resolve()))
import modeling_setup as ms

SEED = ms.RANDOM_STATE
ECD = "ECD_CD_cells_per_mm2"
IMAGE_DATA_DIR = ms.PROCESSED_DIR.parent.parent / "image" / "data" / "processed"
EMBEDDING_PATH = IMAGE_DATA_DIR / "embeddings" / "resnet50_imagenet_fullres.npz"
NB10_SCORES_PATH = ms.PROCESSED_DIR / "beyond_ecd_fold_scores.csv"
REVIEW_IDS = ["NEB-2024-08-16259 R", "NEB-2025-09-17471 R"]
PCA_COMPONENTS = 16

# Protocol Section 4.
RF_PARAMS = dict(
    n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=1,
    max_features="sqrt", class_weight="balanced", random_state=SEED, n_jobs=-1,
)
TREE_PARAMS = dict(
    max_depth=2, min_samples_leaf=10, class_weight="balanced",
    criterion="gini", random_state=SEED,
)

# Protocol Section 3.
EXPECTED_PRIMARY = {"tissues": 551, "donors": 312}
EXPECTED_EVAL = {
    "tissues": 470, "donors": 280,
    "grades": {"A+": 244, "A": 159, "B": 67},
    "years": {2020: 78, 2021: 11, 2022: 54, 2024: 155, 2025: 172},
}

# Provenance of the input data (not part of the protocol; recorded for audit).
for path in [ms.DATA_PATH, ms.DONOR_FOLDS_PATH, EMBEDDING_PATH, NB10_SCORES_PATH]:
    print(f"{path.name}: sha256 {hashlib.sha256(path.read_bytes()).hexdigest()[:16]}…")

cleaned_grading_dataset.xlsx: sha256 0ce9b384a51a4c47…
donor_grouped_folds.csv: sha256 755280ff9d7bd388…
resnet50_imagenet_fullres.npz: sha256 883d4a1723a17be7…
beyond_ecd_fold_scores.csv: sha256 f77147e30b1e6c42…


## Cohort (Protocol Section 3)

In [2]:
def normalize_id(value):
    return " ".join(str(value).split()).replace("NEB- ", "NEB-").casefold()

labelled = ms.load_labelled_data()
folds = ms.get_donor_folds(labelled)

manifest = pd.read_csv(IMAGE_DATA_DIR / "clean_image_manifest.csv")
manifest["Match_ID"] = manifest["Tissue_ID"].map(normalize_id)
manifest = manifest[~manifest["Match_ID"].isin({normalize_id(i) for i in REVIEW_IDS})]

labelled["Match_ID"] = labelled["Tissue_ID"].map(normalize_id)
labelled["Year"] = labelled["Tissue_ID"].str.extract(r"NEB-\s*(\d{4})")[0].astype("Int64")

with_image = labelled.merge(manifest[["Match_ID"]], on="Match_ID", how="inner", validate="one_to_one")
cohort = (
    with_image[with_image["Year"].notna() & (with_image["Year"] != 2023)]
    .merge(folds, on="Tissue_ID", how="left", validate="one_to_one")
    .reset_index(drop=True)
)
cohort["Year"] = cohort["Year"].astype(int)

has_ecd = cohort[ECD].notna().to_numpy()
evaluation = cohort[has_ecd]

primary_desc = {"tissues": len(cohort), "donors": cohort["Donor_Group"].nunique()}
eval_desc = {
    "tissues": len(evaluation),
    "donors": evaluation["Donor_Group"].nunique(),
    "grades": {k: int(v) for k, v in evaluation[ms.TARGET].value_counts().items()},
    "years": {int(k): int(v) for k, v in evaluation["Year"].value_counts().sort_index().items()},
}
print("Primary cohort:", primary_desc)
print("Evaluation set:", eval_desc)
assert primary_desc == EXPECTED_PRIMARY, "Primary cohort differs from protocol"
assert eval_desc == EXPECTED_EVAL, "Evaluation set differs from protocol"
print("\nBoth sets match the protocol.")

Primary cohort: {'tissues': 551, 'donors': 312}
Evaluation set: {'tissues': 470, 'donors': 280, 'grades': {'A+': 244, 'A': 159, 'B': 67}, 'years': {2020: 78, 2021: 11, 2022: 54, 2024: 155, 2025: 172}}

Both sets match the protocol.


In [3]:
cached = np.load(EMBEDDING_PATH, allow_pickle=True)
row_of = {tissue_id: i for i, tissue_id in enumerate(cached["tissue_ids"])}
assert all(t in row_of for t in cohort["Tissue_ID"]), "Missing cached embeddings"
embeddings = cached["embeddings"][[row_of[t] for t in cohort["Tissue_ID"]]]

EMBEDDING_COLUMNS = [f"emb_{i}" for i in range(embeddings.shape[1])]
X = pd.concat([
    cohort[ms.ALL_FEATURES].reset_index(drop=True),
    pd.DataFrame(embeddings, columns=EMBEDDING_COLUMNS),
], axis=1)
y = cohort[ms.TARGET].to_numpy()
years = cohort["Year"].to_numpy()
print("Feature matrix:", X.shape)

Feature matrix: (551, 2072)


## Models (Protocol Section 4)

In [4]:
def make_arm(name):
    if name == "ECD tree":
        return Pipeline([
            ("features", ColumnTransformer([("ecd", "passthrough", [ECD])], remainder="drop")),
            ("classifier", DecisionTreeClassifier(**TREE_PARAMS)),
        ])
    if name == "ECD-only RF":
        branches = [("ecd", "passthrough", [ECD])]
    else:
        branches = [("tabular", ms.make_preprocessor(), ms.ALL_FEATURES)]
        if name == "Hybrid":
            branches.append(("image", Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=PCA_COMPONENTS, random_state=SEED)),
            ]), EMBEDDING_COLUMNS))
    return Pipeline([
        ("features", ColumnTransformer(branches, remainder="drop")),
        ("classifier", RandomForestClassifier(**RF_PARAMS)),
    ])

ARMS = {name: make_arm(name) for name in ["ECD tree", "ECD-only RF", "Tabular", "Hybrid"]}
ECD_ONLY = {"ECD tree", "ECD-only RF"}


def ecd_rule(values):
    # Descriptive only: thresholds inferred from 2023 records (notebook 10, Section 8).
    return np.where(values >= 2500, "A+", np.where(values >= 2250, "A", "B"))

## Validation checks (Protocol Section 5)

All checks must pass before any model is fitted.

In [5]:
fold_splits = {
    repeat: list(ms.iter_donor_folds(folds, cohort["Tissue_ID"], repeat))
    for repeat in range(ms.N_REPEATS)
}

for repeat, splits in fold_splits.items():
    tested = np.sort(np.concatenate([test for _, test in splits]))
    assert np.array_equal(tested, np.arange(len(cohort))), "Each tissue must be tested once per repeat"
    for train, test in splits:
        assert set(cohort.loc[train, "Donor_Group"]).isdisjoint(cohort.loc[test, "Donor_Group"]), "Donor leak"
        assert set(y[test[has_ecd[test]]]) == set(ms.CLASSES), "A held-out fold lacks a grade in the evaluation set"

sizes = np.array([(len(tr), len(te)) for s in fold_splits.values() for tr, te in s])
N_TRAIN, N_TEST = sizes.mean(axis=0)
print("All validation checks passed.")

All validation checks passed.


## Cross-validated predictions

In [6]:
oof = {(arm, r): np.full(len(cohort), None, dtype=object) for arm in ARMS for r in range(ms.N_REPEATS)}
cut_points = []

start = time.time()
for repeat, splits in fold_splits.items():
    for fold, (train, test) in enumerate(splits):
        for arm, pipeline in ARMS.items():
            if arm in ECD_ONLY:
                train_rows, test_rows = train[has_ecd[train]], test[has_ecd[test]]
            else:
                train_rows, test_rows = train, test
            model = clone(pipeline).fit(X.iloc[train_rows], y[train_rows])
            oof[(arm, repeat)][test_rows] = model.predict(X.iloc[test_rows])

            if arm == "ECD tree":
                tree = model.named_steps["classifier"].tree_
                thresholds = sorted(tree.threshold[tree.feature >= 0])
                cut_points.append({"Repeat": repeat, "Fold": fold, "Cut points": [round(t, 1) for t in thresholds]})

print(f"{len(sizes) * len(ARMS)} model fits in {(time.time() - start) / 60:.1f} min")

100 model fits in 1.2 min


## Fold-level metrics on the evaluation set (Protocol Section 6)

In [7]:
METRICS = {
    "Macro F1": lambda t, p: f1_score(t, p, labels=ms.CLASSES, average="macro", zero_division=0),
    "Balanced accuracy": balanced_accuracy_score,
    "Accuracy": accuracy_score,
    "Recall B": lambda t, p: recall_score(t, p, labels=["B"], average="macro", zero_division=0),
}

rule_predictions = np.full(len(cohort), None, dtype=object)
rule_predictions[has_ecd] = ecd_rule(cohort.loc[has_ecd, ECD].to_numpy())

fold_rows = []
for repeat, splits in fold_splits.items():
    for fold, (_, test) in enumerate(splits):
        rows = test[has_ecd[test]]
        predictors = {arm: oof[(arm, repeat)][rows] for arm in ARMS}
        predictors["ECD rule"] = rule_predictions[rows]
        for name, pred in predictors.items():
            fold_rows.append({
                "Model": name, "Repeat": repeat, "Fold": fold, "N": len(rows),
                **{m: fn(y[rows], pred.astype(str)) for m, fn in METRICS.items()},
            })

fold_scores = pd.DataFrame(fold_rows)
MODELS = ["ECD tree", "ECD-only RF", "Tabular", "Hybrid", "ECD rule"]
print("Evaluation-set metrics, mean and SD over 25 folds (ECD-only RF and ECD rule descriptive only):")
print(fold_scores.groupby("Model")[list(METRICS)].agg(["mean", "std"]).reindex(MODELS).round(4).to_string())

Evaluation-set metrics, mean and SD over 25 folds (ECD-only RF and ECD rule descriptive only):
            Macro F1         Balanced accuracy         Accuracy         Recall B        
                mean     std              mean     std     mean     std     mean     std
Model                                                                                   
ECD tree      0.7958  0.0383            0.7680  0.0431   0.8044  0.0308   0.7901  0.1171
ECD-only RF   0.7073  0.0509            0.7114  0.0523   0.7093  0.0491   0.7929  0.1104
Tabular       0.7557  0.0423            0.7533  0.0434   0.7582  0.0403   0.8140  0.1017
Hybrid        0.7886  0.0412            0.7821  0.0445   0.7965  0.0325   0.8089  0.1272
ECD rule      0.7973  0.0373            0.7694  0.0424   0.8064  0.0289   0.7901  0.1171


### Reproducibility check (Protocol Section 4)

Tabular, Hybrid and ECD-only RF fold scores must match notebook 10.

In [8]:
saved = pd.read_csv(NB10_SCORES_PATH)
name_in_nb10 = {"ECD-only RF": "ECD-only", "Tabular": "Tabular", "Hybrid": "Hybrid"}
for ours_name, saved_name in name_in_nb10.items():
    ours = fold_scores[fold_scores["Model"] == ours_name].set_index(["Repeat", "Fold"])[list(METRICS)]
    theirs = saved[saved["Model"] == saved_name].set_index(["Repeat", "Fold"])[list(METRICS)]
    max_diff = (ours.sort_index() - theirs.sort_index()).abs().to_numpy().max()
    print(f"{ours_name}: max absolute difference from notebook 10 = {max_diff:.2e}")
    assert max_diff < 1e-9, f"{ours_name} differs from notebook 10: stop and record in the deviations log"

ECD-only RF: max absolute difference from notebook 10 = 0.00e+00
Tabular: max absolute difference from notebook 10 = 0.00e+00
Hybrid: max absolute difference from notebook 10 = 1.11e-16


## Primary comparisons P1 and P2 (Protocol Section 7)

In [9]:
def corrected_ttest(differences):
    d = np.asarray(differences, dtype=float)
    n = len(d)
    se = np.sqrt((1 / n + N_TEST / N_TRAIN) * d.var(ddof=1))
    t = d.mean() / se
    p = 2 * stats.t.sf(abs(t), df=n - 1)
    half = stats.t.ppf(0.975, df=n - 1) * se
    return d.mean(), d.mean() - half, d.mean() + half, t, p


def holm(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (len(p_values) - rank) * p_values[idx])
        adjusted[idx] = min(running, 1.0)
    return adjusted


wide = fold_scores.pivot_table(index=["Repeat", "Fold"], columns="Model", values=list(METRICS))
COMPARISONS = {"P1": ("Tabular", "ECD tree"), "P2": ("Hybrid", "ECD tree")}

rows = []
for label, (model, baseline) in COMPARISONS.items():
    for metric in METRICS:
        d = wide[(metric, model)] - wide[(metric, baseline)]
        mean, low, high, t, p = corrected_ttest(d)
        rows.append({
            "Comparison": f"{label}: {model} − {baseline}",
            "Role": "Primary" if metric == "Macro F1" else "Secondary",
            "Metric": metric,
            "Model": wide[(metric, model)].mean(),
            "ECD tree": wide[(metric, baseline)].mean(),
            "Difference": mean, "CI low": low, "CI high": high, "t": t, "p": p,
            "Folds improved": f"{(d > 0).sum()}/{len(d)}",
        })
results = pd.DataFrame(rows)

primary_mask = results["Role"] == "Primary"
results["p (Holm)"] = np.nan
results.loc[primary_mask, "p (Holm)"] = holm(results.loc[primary_mask, "p"])

print("Primary (macro F1; Holm across P1 and P2; CIs not multiplicity-adjusted):")
print(results[primary_mask].drop(columns="Role").round(4).to_string(index=False))
print("\nSecondary (unadjusted):")
print(results[~primary_mask].drop(columns=["Role", "p (Holm)"]).round(4).to_string(index=False))

Primary (macro F1; Holm across P1 and P2; CIs not multiplicity-adjusted):
            Comparison   Metric  Model  ECD tree  Difference  CI low  CI high       t      p Folds improved  p (Holm)
P1: Tabular − ECD tree Macro F1 0.7557    0.7958     -0.0402  -0.084   0.0037 -1.8913 0.0707           2/25    0.1414
 P2: Hybrid − ECD tree Macro F1 0.7886    0.7958     -0.0073  -0.046   0.0314 -0.3878 0.7016          11/25    0.7016

Secondary (unadjusted):
            Comparison            Metric  Model  ECD tree  Difference  CI low  CI high       t      p Folds improved
P1: Tabular − ECD tree Balanced accuracy 0.7533    0.7680     -0.0147 -0.0516   0.0221 -0.8245 0.4178           6/25
P1: Tabular − ECD tree          Accuracy 0.7582    0.8044     -0.0462 -0.0909  -0.0014 -2.1285 0.0438           2/25
P1: Tabular − ECD tree          Recall B 0.8140    0.7901      0.0239 -0.0243   0.0721  1.0233 0.3164           7/25
 P2: Hybrid − ECD tree Balanced accuracy 0.7821    0.7680      0.0141 -0.0251  

## Descriptive results (Protocol Section 6)

In [10]:
for other in ["ECD-only RF", "ECD rule"]:
    d = wide[("Macro F1", "ECD tree")] - wide[("Macro F1", other)]
    print(f"ECD tree − {other}: mean fold-level macro F1 difference {d.mean():+.4f} "
          f"(tree higher in {(d > 0).sum()}/{len(d)} folds)")

cuts = pd.DataFrame(cut_points)
all_cuts = np.concatenate(cuts["Cut points"].map(np.array).to_numpy())
print(f"\nCut points learned by the ECD tree across 25 training folds "
      f"({cuts['Cut points'].map(len).value_counts().sort_index().to_dict()} folds by number of cut points):")
print(f"  all cut points: min {all_cuts.min():.0f}, median {np.median(all_cuts):.0f}, max {all_cuts.max():.0f}")
lower = cuts["Cut points"].map(min)
upper = cuts["Cut points"].map(max)
print(f"  lowest cut per fold: {lower.min():.0f}–{lower.max():.0f} (median {lower.median():.0f})")
print(f"  highest cut per fold: {upper.min():.0f}–{upper.max():.0f} (median {upper.median():.0f})")
print(cuts.to_string(index=False))

ECD tree − ECD-only RF: mean fold-level macro F1 difference +0.0886 (tree higher in 25/25 folds)
ECD tree − ECD rule: mean fold-level macro F1 difference -0.0015 (tree higher in 0/25 folds)

Cut points learned by the ECD tree across 25 training folds ({2: 25} folds by number of cut points):
  all cut points: min 2232, median 2363, max 2503
  lowest cut per fold: 2232–2232 (median 2232)
  highest cut per fold: 2494–2503 (median 2497)
 Repeat  Fold       Cut points
      0     0 [2232.0, 2497.0]
      0     1 [2232.0, 2497.0]
      0     2 [2232.0, 2497.0]
      0     3 [2232.0, 2503.0]
      0     4 [2232.0, 2497.0]
      1     0 [2232.0, 2497.0]
      1     1 [2232.0, 2497.0]
      1     2 [2232.0, 2497.0]
      1     3 [2232.0, 2497.0]
      1     4 [2232.0, 2503.0]
      2     0 [2232.0, 2497.0]
      2     1 [2232.0, 2497.0]
      2     2 [2232.0, 2497.0]
      2     3 [2232.0, 2497.0]
      2     4 [2232.0, 2503.0]
      3     0 [2232.0, 2503.0]
      3     1 [2232.0, 2497.0]
     

In [11]:
eval_rows = np.flatnonzero(has_ecd)
y_eval = np.tile(y[eval_rows], ms.N_REPEATS)
pooled = {arm: np.concatenate([oof[(arm, r)][eval_rows] for r in range(ms.N_REPEATS)]).astype(str) for arm in ARMS}
pooled["ECD rule"] = np.tile(rule_predictions[eval_rows], ms.N_REPEATS).astype(str)

print("Per-grade precision and recall on the evaluation set, pooled over repeats:")
print(pd.concat({
    name: pd.DataFrame({
        "Precision": precision_score(y_eval, pooled[name], labels=ms.CLASSES, average=None, zero_division=0),
        "Recall": recall_score(y_eval, pooled[name], labels=ms.CLASSES, average=None, zero_division=0),
    }, index=ms.CLASSES)
    for name in MODELS
}, axis=1).round(3).to_string())

Per-grade precision and recall on the evaluation set, pooled over repeats:
    ECD tree        ECD-only RF          Tabular           Hybrid         ECD rule       
   Precision Recall   Precision Recall Precision Recall Precision Recall Precision Recall
A+     0.748  0.988       0.761  0.785     0.784  0.846     0.809  0.890     0.749  0.992
A      0.884  0.528       0.594  0.558     0.676  0.599     0.745  0.649     0.894  0.528
B      1.000  0.791       0.773  0.794     0.830  0.818     0.860  0.806     1.000  0.791


### Per-year results (descriptive)

2021 (11 tissues) is too small to interpret.

In [12]:
macro_f1 = METRICS["Macro F1"]
years_eval = np.tile(years[eval_rows], ms.N_REPEATS)
eval_years = sorted(EXPECTED_EVAL["years"])

by_year, leave_out = [], []
for label, (model, baseline) in COMPARISONS.items():
    for year in eval_years:
        sel = years_eval == year
        model_score = macro_f1(y_eval[sel], pooled[model][sel])
        base_score = macro_f1(y_eval[sel], pooled[baseline][sel])
        by_year.append({
            "Comparison": label, "Year": year, "Tissues": EXPECTED_EVAL["years"][year],
            "Model": model_score, "ECD tree": base_score, "Difference": model_score - base_score,
        })
        keep = years_eval != year
        leave_out.append({
            "Comparison": label, "Excluded year": year,
            "Difference": macro_f1(y_eval[keep], pooled[model][keep]) - macro_f1(y_eval[keep], pooled[baseline][keep]),
        })

by_year, leave_out = pd.DataFrame(by_year), pd.DataFrame(leave_out)
overall_pooled = {
    label: macro_f1(y_eval, pooled[model]) - macro_f1(y_eval, pooled[baseline])
    for label, (model, baseline) in COMPARISONS.items()
}

for label, (model, _) in COMPARISONS.items():
    print(f"{label} ({model} − ECD tree): overall pooled difference {overall_pooled[label]:+.4f}")
    print(by_year[by_year["Comparison"] == label].drop(columns="Comparison").round(4).to_string(index=False))
    print(leave_out[leave_out["Comparison"] == label].drop(columns="Comparison").round(4).to_string(index=False), "\n")

P1 (Tabular − ECD tree): overall pooled difference -0.0411
 Year  Tissues  Model  ECD tree  Difference
 2020       78 0.7378    0.7240      0.0138
 2021       11 0.7593    0.7554      0.0038
 2022       54 0.7826    0.7057      0.0769
 2024      155 0.7500    0.7916     -0.0416
 2025      172 0.7455    0.8289     -0.0834
 Excluded year  Difference
          2020     -0.0482
          2021     -0.0433
          2022     -0.0533
          2024     -0.0359
          2025     -0.0127 

P2 (Hybrid − ECD tree): overall pooled difference -0.0078
 Year  Tissues  Model  ECD tree  Difference
 2020       78 0.7483    0.7240      0.0243
 2021       11 0.7805    0.7554      0.0251
 2022       54 0.7800    0.7057      0.0743
 2024      155 0.7827    0.7916     -0.0089
 2025      172 0.7937    0.8289     -0.0353
 Excluded year  Difference
          2020     -0.0114
          2021     -0.0106
          2022     -0.0155
          2024     -0.0039
          2025      0.0110 



## Interpretation (Protocol Section 8)

The rules are applied mechanically to the results above.

In [13]:
primary = results[primary_mask].set_index("Comparison")
b_recall = results[results["Metric"] == "Recall B"].set_index("Comparison")["Difference"]

verdicts = {}
for comparison, row in primary.iterrows():
    label = comparison.split(":")[0]
    if row["Difference"] > 0 and row["p (Holm)"] < 0.05:
        verdicts[label] = "improved"
    elif row["Difference"] < 0 and row["p (Holm)"] < 0.05:
        verdicts[label] = "worse"
    else:
        verdicts[label] = "inconclusive"

    print(f"{comparison}: macro F1 {row['Difference']:+.4f} [{row['CI low']:+.4f}, {row['CI high']:+.4f}], "
          f"p = {row['p']:.4f}, Holm p = {row['p (Holm)']:.4f} -> {verdicts[label]}")

    b_difference = b_recall[comparison]
    print(f"  Grade B recall qualifier: {'TRIGGERED' if b_difference < -0.02 else 'not triggered'} "
          f"(mean B recall difference {b_difference:+.4f}; threshold −0.02)")

    flagged = leave_out.loc[(leave_out["Comparison"] == label) & (leave_out["Difference"] <= 0), "Excluded year"].tolist()
    year_dependent = overall_pooled[label] > 0 and bool(flagged)
    print(f"  Per-year qualifier: {'TRIGGERED, depends on ' + str(flagged) if year_dependent else 'not triggered'}")

worse = [label for label, v in verdicts.items() if v == "worse"]
if worse:
    answer = f"Models performed worse than a fitted ECD-only threshold model: {', '.join(COMPARISONS[l][0] for l in worse)}"
elif "improved" in verdicts.values():
    answer = "Evidence of improvement beyond a fitted ECD-only threshold model (follow-up analysis)"
else:
    answer = "Improvement beyond ECD alone not established"

print(f"\nAnswer to the research question: {answer}.")
print("Reported alongside notebook 10 (improvement over an ECD-only Random Forest); "
      "this is a follow-up motivated by notebook 10, not independent confirmation.")

P1: Tabular − ECD tree: macro F1 -0.0402 [-0.0840, +0.0037], p = 0.0707, Holm p = 0.1414 -> inconclusive
  Grade B recall qualifier: not triggered (mean B recall difference +0.0239; threshold −0.02)
  Per-year qualifier: not triggered
P2: Hybrid − ECD tree: macro F1 -0.0073 [-0.0460, +0.0314], p = 0.7016, Holm p = 0.7016 -> inconclusive
  Grade B recall qualifier: not triggered (mean B recall difference +0.0189; threshold −0.02)
  Per-year qualifier: not triggered

Answer to the research question: Improvement beyond ECD alone not established.
Reported alongside notebook 10 (improvement over an ECD-only Random Forest); this is a follow-up motivated by notebook 10, not independent confirmation.


In [14]:
fold_scores.to_csv(ms.PROCESSED_DIR / "fitted_ecd_fold_scores.csv", index=False)
results.to_csv(ms.PROCESSED_DIR / "fitted_ecd_results.csv", index=False)
cuts.to_csv(ms.PROCESSED_DIR / "fitted_ecd_cut_points.csv", index=False)
print("Saved fold scores, comparison results and cut points to", ms.PROCESSED_DIR)

Saved fold scores, comparison results and cut points to C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\grading\data\processed


## Deviations Log

Record any rerun or departure from the protocol here, with date and reason, before reporting results.

| Date | Change | Reason |
|---|---|---|
| — | None | — |